## let's wrangle some of this data

In [12]:
import pandas as pd
import seaborn as sns
import numpy as np

In [13]:
df = pd.read_csv("lut_with_ontology_metadata.csv")
print(df.columns)
df.head()

Index(['clean_label', 'SNOMED_ID', 'UBERON_ID', 'FMA_ID', 'Definition',
       'synonyms', 'Source', 'fiber_category', 'atlas_composite_flag',
       'laterality', 'Ontology_cross_reference', 'uberon_label',
       'uberon_definition', 'uberon_synonyms_exact', 'uberon_synonyms_related',
       'uberon_synonyms_broad', 'uberon_synonyms_narrow', 'uberon_xrefs',
       'uberon_snomed_xrefs', 'uberon_fma_xrefs', 'snomed_id_validated'],
      dtype='object')


,clean_label,SNOMED_ID,UBERON_ID,FMA_ID,Definition,synonyms,Source,fiber_category,atlas_composite_flag,laterality,...,uberon_label,uberon_definition,uberon_synonyms_exact,uberon_synonyms_related,uberon_synonyms_broad,uberon_synonyms_narrow,uberon_xrefs,uberon_snomed_xrefs,uberon_fma_xrefs,snomed_id_validated
0,anterior commissure,SNOMED:62872008,UBERON:0000935,fma61961,Myelinated nerve fibers passing transversely t...,AC ; Anterior commissure ; Structure of anteri...,TRACULA ; MGH_HCP ; Bullock_2022 ; XTRACT ; Tr...,commissural,False,midline,...,anterior commissure,A bundle of myelinated nerve fibers passing tr...,anterior cerebral commissure | commissura ante...,AC | anterior commissural nucleus | commissura...,NaN,NaN,BAMS:AC | BAMS:Ac | BAMS:ac | BAMS:AG | BIRNLE...,SCTID:279313001 | SCTID:369119009,FMA:61961,False
1,corpus callosum,SNOMED:362354006,UBERON:0002336,fma86464,The largest commissural white matter structure...,Entire corpus callosum,ICBM-DTI-81 ; Bullock_2022,commissural,False,midline,...,corpus callosum,White matter structure containing massive numb...,NaN,NaN,NaN,NaN,BAMS:CC | BAMS:cc | BIRNLEX:1087 | BM:Tel-CC |...,SCTID:362354006,FMA:86464,True
2,cingulum,SNOMED:37035000,UBERON:0003961,fma260761,Long C-shaped association bundle running adjac...,Cingulate fasciculus ; Cingulum ; Structure of...,TractSeg ; ICBM-DTI-81 ; Bullock_2022,association,False,bilateral,...,cingulum of brain,The white matter fiber bundle that projects fr...,cingulum bundle,cingulum of telencephalon | neuraxis cingulum,cingulum,NaN,BAMS:cg | DHBA:10572 | EMAPA:37828 | FMA:26076...,SCTID:369084007,FMA:260761 | FMA:83869,False
3,external capsule,SNOMED:279300007,UBERON:0004545,fma61959,A sheet of white matter lateral to the putamen...,External capsule ; Structure of external capsu...,ICBM-DTI-81 ; Bullock_2022,association,False,bilateral,...,external capsule of telencephalon,Any of the series of white matter fiber tracts...,NaN,capsula externa | corpus callosum external cap...,brain external capsule | external capsule,NaN,BAMS:EC | BAMS:ec | DHBA:10573 | DMBA:17765 | ...,SCTID:279300007,FMA:61959,True
4,extreme capsule,SNOMED:279301006,UBERON:0014528,fma61960,White matter lateral to the claustrum and medi...,EmC ; band of Baillarger ; capsula extrema,TRACULA ; MGH_HCP ; Bullock_2022,association,False,bilateral,...,extreme capsule,Thin band of fibers separating the claustrum f...,NaN,band of Baillarger | capsula extrema,NaN,NaN,BAMS:ex | BAMS:exc | DHBA:10574 | FMA:61960 | ...,SCTID:279301006,FMA:61960,True


In [14]:
# --- 1. Helper function to merge text/list columns cleanly ---
def merge_and_clean(row, target_col, columns_to_add, separator='; '):
    """
    Gathers existing items from target_col and appends new items from columns_to_add.
    Handles both lists and delimited strings, removing duplicates.
    """
    items = []
    
    # Collect all sources to merge
    all_sources = [target_col] + columns_to_add
    
    for source in all_sources:
        val = row[source] if source in row.index else None
        if pd.isna(val) or val is None:
            continue
            
        # If it's already a list/array, extend our list
        if isinstance(val, (list, set, tuple, np.ndarray)):
            items.extend([str(i).strip() for i in val if str(i).strip()])
        # If it's a string, split it by common delimiters
        elif isinstance(val, str):
            # Split by semicolon, comma, or pipe if they exist
            split_char = separator if separator in val else (',' if ',' in val else '|')
            items.extend([i.strip() for i in val.split(split_char) if i.strip()])
            
    # Remove duplicates while preserving order
    unique_items = []
    for item in items:
        if item not in unique_items:
            unique_items.append(item)
            
    # Return as a clean delimited string (or return unique_items if you prefer list format)
    return separator.join(unique_items) if unique_items else np.nan


# --- 2. Run the pipeline ---
# Ensure your target columns exist (initialize if they don't)
if 'Ontology_cross_reference' not in df.columns:
    df['Ontology_cross_reference'] = np.nan
if 'synonyms' not in df.columns:
    df['synonyms'] = np.nan

# Define the source columns we want to extract from
id_cols_to_merge = ['uberon_xrefs', 'uberon_snomed_xrefs', 'uberon_fma_xrefs']
syn_cols_to_merge = ['uberon_synonyms_exact', 'uberon_synonyms_broad']

# Apply the merge row by row
df['Ontology_cross_reference'] = df.apply(
    lambda r: merge_and_clean(r, 'Ontology_cross_reference', id_cols_to_merge), axis=1
)

df['synonyms'] = df.apply(
    lambda r: merge_and_clean(r, 'synonyms', syn_cols_to_merge), axis=1
)

# --- 3. Clean up the DataFrame ---
# Columns we want to delete entirely based on your request
columns_to_delete = [
    'uberon_label', 
    'uberon_definition', 
    'uberon_synonyms_narrow', 
    'uberon_synonyms_exact', 
    'uberon_synonyms_broad', 
    'uberon_xrefs', 
    'uberon_snomed_xrefs', 
    'uberon_fma_xrefs'
]

# Drop the columns if they exist in your DataFrame
df.drop(columns=[col for col in columns_to_delete if col in df.columns], inplace=True)

In [15]:
df.head()

,clean_label,SNOMED_ID,UBERON_ID,FMA_ID,Definition,synonyms,Source,fiber_category,atlas_composite_flag,laterality,Ontology_cross_reference,uberon_synonyms_related,snomed_id_validated
0,anterior commissure,SNOMED:62872008,UBERON:0000935,fma61961,Myelinated nerve fibers passing transversely t...,AC; Anterior commissure; Structure of anterior...,TRACULA ; MGH_HCP ; Bullock_2022 ; XTRACT ; Tr...,commissural,False,midline,BAMS:AC; BAMS:AG; BAMS:Ac; BAMS:ac; BIRNLEX:15...,AC | anterior commissural nucleus | commissura...,False
1,corpus callosum,SNOMED:362354006,UBERON:0002336,fma86464,The largest commissural white matter structure...,Entire corpus callosum,ICBM-DTI-81 ; Bullock_2022,commissural,False,midline,BAMS:CC; BAMS:cc; BIRNLEX:1087; BM:Tel-CC; BTO...,NaN,True
2,cingulum,SNOMED:37035000,UBERON:0003961,fma260761,Long C-shaped association bundle running adjac...,Cingulate fasciculus; Cingulum; Structure of c...,TractSeg ; ICBM-DTI-81 ; Bullock_2022,association,False,bilateral,BAMS:cg; DHBA:10572; EMAPA:37828; FMA:260761; ...,cingulum of telencephalon | neuraxis cingulum,False
3,external capsule,SNOMED:279300007,UBERON:0004545,fma61959,A sheet of white matter lateral to the putamen...,External capsule; Structure of external capsul...,ICBM-DTI-81 ; Bullock_2022,association,False,bilateral,BAMS:EC; BAMS:ec; DHBA:10573; DMBA:17765; EMAP...,capsula externa | corpus callosum external cap...,True
4,extreme capsule,SNOMED:279301006,UBERON:0014528,fma61960,White matter lateral to the claustrum and medi...,EmC; band of Baillarger; capsula extrema,TRACULA ; MGH_HCP ; Bullock_2022,association,False,bilateral,BAMS:ex; BAMS:exc; DHBA:10574; FMA:61960; HBA:...,band of Baillarger | capsula extrema,True


In [ ]:
'''df.to_csv('extract_ontology_data.tsv', sep='\t', index=False)
print("Transformation complete! File saved as 'extract_ontology_data.tsv'")
'''
df.to_excel('processed_ontology_data.xlsx', index=False)
print("Saved as Excel!")

Saved as Excel! This will upload to Google Drive perfectly.
